# Advanced Financial and Macro Econometrics Assignment 3

In [1]:
import numpy as np
import pandas as pd
from arch import arch_model

In [2]:
df = pd.read_excel(r"C:\Users\victo\Downloads\DanishStocks.xlsx")

In [3]:
df.head()

,Unnamed: 0,Date,ALK-B,ALMB,AMBU-B,BAVA,BNORDIK-CSE,BO,CARL-A,CARL-B,...,SPNO,SYDB,TIV,TOP,TRYG,UIE,VELO,VJBA,VWS,ZEAL
0,1,2011-01-03,323.5,13.0,8.05,195.816,144.0,58.0,582.0,576.0,...,42.082,152.9,329.6,73.90,52.30,600,1.30,11.789,186.4,72.0
1,2,2011-01-04,324.0,13.1,8.25,200.857,141.5,59.0,578.5,571.0,...,42.082,154.4,328.0,74.00,52.70,600,1.31,11.789,185.2,71.0
2,3,2011-01-05,320.0,12.8,8.20,198.918,141.0,59.0,573.0,571.5,...,42.082,154.5,332.0,74.05,52.54,600,1.33,11.618,179.2,72.0
3,4,2011-01-06,316.0,12.7,8.15,201.633,141.0,58.5,588.0,578.5,...,42.435,153.5,334.5,74.55,54.44,604,1.33,11.618,179.6,74.0
4,5,2011-01-07,320.0,12.3,8.10,202.020,142.0,58.0,568.0,555.0,...,42.789,152.0,332.8,74.50,53.90,615,1.31,11.704,176.7,74.5


In [4]:
df["retALMB"] = 100 * (np.log(df["ALMB"]) - np.log(df["ALMB"].shift(1)))
df["retBO"] = 100 * (np.log(df["BO"]) - np.log(df["BO"].shift(1)))
df["retCHR"] = 100 * (np.log(df["CHR"]) - np.log(df["CHR"].shift(1)))
new_df = df[["Date", "retALMB", "retBO", "retCHR"]]
new_df = new_df.dropna()

In [5]:
coefficients = []
cond_vol = []
std_resids = []
models = []

In [6]:
for i in range(1, 4):
    model = arch_model(new_df.iloc[:, i], vol="Garch", p=1, q=1)
    results = model.fit(disp="off")
    coefficients.append(results.params)
    cond_vol.append(results.conditional_volatility)
    std_resids.append(results.std_resid)
    models.append(model)

In [7]:
stock_names = ["ALMB", "BO", "CHR"]
date_index = new_df["Date"].values

# Conditional volatilities
cond_vol_df = pd.DataFrame({name: vol.values for name, vol in zip(stock_names, cond_vol)}, index=new_df["Date"])
cond_vol_df.index.name = "Date"

# Standardized residuals
std_resids_df = pd.DataFrame({name: resid.values for name, resid in zip(stock_names, std_resids)}, index=new_df["Date"])
std_resids_df.index.name = "Date"

# Extract parameter names from the first model
param_names = coefficients[0].index

# Build coefficients DataFrame
coeff_df = pd.DataFrame({name: coef.values for name, coef in zip(stock_names, coefficients)}, index=param_names)
coeff_df.index.name = "Parameter"

In [15]:
# calculate the constant conditional correlation matrix (CCC) R:
R = std_resids_df.transpose().dot(std_resids_df).div(len(std_resids_df))

In [17]:
coeff_df

,ALMB,BO,CHR
Parameter,,,
mu,0.060987,-0.057625,0.080956
omega,0.937152,1.397552,0.291691
alpha[1],0.204774,0.414630,0.082809
beta[1],0.468231,0.585370,0.793107
